In [ ]:
import os
import nltk

In [ ]:
!pip install git+https://github.com/openai/whisper.git
!pip install language-tool-python
!apt-get update
!pip install eyed3 pyAudioAnalysis
!apt-get install openjdk-17-jdk -y
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-17-openjdk-amd64"
!java -version
nltk.download('averaged_perceptron_tagger_eng')
nltk.download('averaged_perceptron_tagger')
nltk.download('punkt_tab')
nltk.download('punkt')

In [ ]:
from pyAudioAnalysis import audioBasicIO, ShortTermFeatures
from nltk.corpus import stopwords
from collections import Counter
from google.colab import files
from typing import List, Tuple
import language_tool_python
import subprocess
import pandas as pd
import numpy as np
import whisper
import librosa
import json
import math
import sys
import cv2
import re

In [ ]:
video_path = '/content/Video Interviewing Example.mp4'
audio_path = "audio.wav"
subprocess.run(["ffmpeg", "-i", video_path, "-q:a", "0", "-map", "a", audio_path])

CompletedProcess(args=['ffmpeg', '-i', '/content/Video Interviewing Example.mp4', '-q:a', '0', '-map', 'a', 'audio.wav'], returncode=1)

In [ ]:
model = whisper.load_model("medium")  # "base" faster, "medium" more accuracy
tool = language_tool_python.LanguageTool('en-US')

100%|█████████████████████████████████████| 1.42G/1.42G [00:17<00:00, 84.9MiB/s]


In [ ]:
result = model.transcribe(
    audio_path,
    language='en',
    condition_on_previous_text=False,
    word_timestamps=True,
    verbose=False,
)

with open("transcript.json", "w") as f:
    json.dump(result, f)

full_text = result['text']

/usr/local/lib/python3.12/dist-packages/whisper/transcribe.py:132: UserWarning: FP16 is not supported on CPU; using FP32 instead
  warnings.warn("FP16 is not supported on CPU; using FP32 instead")
100%|██████████| 16221/16221 [07:31<00:00, 35.95frames/s]


In [ ]:
# Load audio
Fs, x = audioBasicIO.read_audio_file("/content/audio.wav")
x = audioBasicIO.stereo_to_mono(x)

# window = 50 ms, step = 25 ms
win = int(0.050 * Fs)
step = int(0.025 * Fs)
F, f_names = ShortTermFeatures.feature_extraction(x, Fs, win, step)

try:
    energy_idx = f_names.index('energy')
except ValueError:
    print("Available feature names:", f_names)
    energy_idx = 1

energy_frames = F[energy_idx, :].astype(float)
energy_db = 10.0 * np.log10(np.maximum(energy_frames, 1e-12))

n_frames = energy_frames.shape[0]
times = np.arange(n_frames) * (step / Fs)

In [ ]:
# Loudness (Calm / Balanced / Loud)

mean_energy_db = float(np.mean(energy_db))
median_energy_db = float(np.median(energy_db))
std_energy_db = float(np.std(energy_db))

def _map_db_to_0_100(mean_db, ref_db=median_energy_db, min_offset=-35.0, max_offset=20.0):
    min_db = ref_db + min_offset
    max_db = ref_db + max_offset
    score = (mean_db - min_db) / (max_db - min_db) * 100.0
    return float(max(0.0, min(100.0, score)))

avg_loudness = _map_db_to_0_100(mean_energy_db)

if avg_loudness < 30:
    loud_label = "Calm / Soft-spoken"
elif avg_loudness < 60:
    loud_label = "Balanced"
else:
    loud_label = "Loud"

print(f"Loudness score: {avg_loudness:.1f} → {loud_label}")


# Expressiveness (Flat / Moderate / High) (dynamicness)
variation = float(max(0.0, min(100.0, (std_energy_db / 8.0) * 100.0)))

if variation < 10:
    expressiveness = "Flat / Monotone"
elif variation < 40:
    expressiveness = "Moderately Expressive"
else:
    expressiveness = "Highly Expressive"

print(f"Expressiveness score: {variation:.1f} → {expressiveness}")

Loudness score: 36.5 → Balanced
Expressiveness score: 100.0 → Highly Expressive


In [ ]:
# Pace (Slow / Normal / Fast)

duration = len(x) / Fs if Fs > 0 else 0.0
if n_frames >= 3 and duration > 0:
    peak_threshold = np.mean(energy_frames) + 0.1 * (np.max(energy_frames) - np.min(energy_frames))
    interior = (energy_frames[1:-1] > energy_frames[:-2]) & (energy_frames[1:-1] > energy_frames[2:]) & (energy_frames[1:-1] > peak_threshold)
    peak_indices = np.where(interior)[0] + 1
    n_peaks = peak_indices.shape[0]
    speech_rate = n_peaks / duration
else:
    speech_rate = 0.0

if speech_rate < 1.5:
    rate_label = "Slow / Thoughtful"
elif speech_rate < 3.0:
    rate_label = "Normal pace"
else:
    rate_label = "Fast-paced / Dynamic"

print(f"Speech rate: {speech_rate:.2f} peaks/sec → {rate_label}")


Speech rate: 0.35 peaks/sec → Slow / Thoughtful


In [ ]:
# Vocabulary Richness Score
words = nltk.word_tokenize(full_text.lower())
words = [w for w in words if re.match(r'[a-z]+$', w)]

# --- Type–Token Ratio ---
unique_words = set(words)
ttr = len(unique_words) / len(words) if words else 0

# --- Lexical Density ---
pos_tags = nltk.pos_tag(words)
content_words = [w for w, t in pos_tags if t.startswith(('N','V','J','R'))]  # Nouns, Verbs, Adj, Adv
lexical_density = len(content_words) / len(words) if words else 0

# Combined Lexical Richness Score from TTR and Lexical Density
ttr = float(ttr)
lexden = float(lexical_density)

# 1) Weighted average (equal weights by default)
w1 = w2 = 1.0
score_avg = 100.0 * ((w1 * ttr + w2 * lexden) / (w1 + w2))

# 2) Geometric mean (balance-emphasizing)
score_geo = 100.0 * (ttr * lexden) ** 0.5

# 3) Harmonic mean / F1-style (strongly penalizes imbalance)
score_f1 = 100.0 * (2 * ttr * lexden) / (ttr + lexden) if (ttr + lexden) > 0 else 0.0

# Determine vocabulary richness flag
if score_f1 >= 80:
    vocabulary_richness_flag = 'Outstanding – highly varied and content-rich speech'
elif score_f1 >= 60:
    vocabulary_richness_flag = 'Strong – clear and fairly informative with good variety'
elif score_f1 >= 40:
    vocabulary_richness_flag = 'Developing – somewhat repetitive or lacking detail'
else:
    vocabulary_richness_flag = 'Limited – simple wording with little variety or depth'


print(f"🧠 Vocabulary Richness Score: {score_f1:.1f}/100 →", vocabulary_richness_flag)

🧠 Vocabulary Richness Score: 51.5/100 → Developing – somewhat repetitive or lacking detail


In [ ]:
print("=== Analysis Results ===")

print(f"  Vocabulary Richness Score: {score_f1:.1f}/100")
print(f"\n  Mean energy (dB): {mean_energy_db:.2f}, median {median_energy_db:.2f}")
print(f"  Std energy (dB): {std_energy_db:.2f}")
print(f"  Estimated onset-peaks: {n_peaks if 'n_peaks' in locals() else 0}, duration: {duration:.2f}s, speech_rate (peaks/sec): {speech_rate:.2f}")

print("\n=== Speaking Style Profile ===")

print(f"  Loudness: {loud_label} → {avg_loudness:.0f}/100%")
print(f"  Expressiveness score: {expressiveness} → {variation:.0f}")
print(f"  Pace: {rate_label} (approx. → {speech_rate:.2f} onsets/sec)")
print(f"  Vocabulary: {vocabulary_richness_flag}")

=== Analysis Results ===
  Vocabulary Richness Score: 51.5/100

  Mean energy (dB): -49.11, median -34.21
  Std energy (dB): 31.97
  Estimated onset-peaks: 57, duration: 162.21s, speech_rate (peaks/sec): 0.35

=== Speaking Style Profile ===
  Loudness: Balanced → 37/100%
  Expressiveness score: Highly Expressive → 100
  Pace: Slow / Thoughtful (approx. → 0.35 onsets/sec)
  Vocabulary: Developing – somewhat repetitive or lacking detail
